# Discovery + Silver: `crm.opportunity_contacts`

Tabla puente N:N entre `opportunities` y `contacts`. Clave compuesta (`opportunity_id`, `contact_id`).

In [1]:
import sys
sys.path.append("/home/jovyan/work/src")

import pandas as pd
from utils.db import get_engine

engine = get_engine()
df = pd.read_sql("SELECT * FROM bronze.crm__opportunity_contacts", engine)
df.shape

(6000, 6)

## 1. Forma general

In [2]:
print(df.dtypes)
df.head()

opportunity_id            object
contact_id                object
role                      object
_source_file              object
_ingested_at      datetime64[ns]
_dag_run_id               object
dtype: object


,opportunity_id,contact_id,role,_source_file,_ingested_at,_dag_run_id
0,OPP-0001114,CON-0013934,decision_maker,crm/opportunity_contacts.csv,2026-07-17 15:16:33.181082,manual__2026-07-17T15:16:30+00:00
1,OPP-0000426,CON-0013348,decision_maker,crm/opportunity_contacts.csv,2026-07-17 15:16:33.181082,manual__2026-07-17T15:16:30+00:00
2,OPP-0001010,CON-0014170,influencer,crm/opportunity_contacts.csv,2026-07-17 15:16:33.181082,manual__2026-07-17T15:16:30+00:00
3,OPP-0000393,CON-0008739,end_user,crm/opportunity_contacts.csv,2026-07-17 15:16:33.181082,manual__2026-07-17T15:16:30+00:00
4,OPP-0002735,CON-0002228,technical,crm/opportunity_contacts.csv,2026-07-17 15:16:33.181082,manual__2026-07-17T15:16:30+00:00


## 2. Nulos, duplicados de la clave compuesta e integridad referencial

In [3]:
print("Nulos por columna:")
print(df.isna().sum())
print()
print("Pares (opportunity_id, contact_id) duplicados:", df.duplicated(subset=["opportunity_id", "contact_id"]).sum())

opportunities = pd.read_sql("SELECT opportunity_id FROM silver.crm__opportunities", engine)
contacts = pd.read_sql("SELECT contact_id FROM silver.crm__contacts", engine)
print("opportunity_id huerfanos:", (~df["opportunity_id"].isin(opportunities["opportunity_id"])).sum())
print("contact_id huerfanos:", (~df["contact_id"].isin(contacts["contact_id"])).sum())

Nulos por columna:
opportunity_id    0
contact_id        0
role              0
_source_file      0
_ingested_at      0
_dag_run_id       0
dtype: int64

Pares (opportunity_id, contact_id) duplicados: 0
opportunity_id huerfanos: 0
contact_id huerfanos: 0


## 3. `role`: valores

In [4]:
print("role:")
print(df["role"].value_counts())

role:
role
influencer        1253
end_user          1232
financial         1200
technical         1159
decision_maker    1156
Name: count, dtype: int64


## 4. Conclusion

Tabla limpia (sin nulos, sin pares duplicados, 0 FKs huerfanas). Solo estandarizacion de `role`.

## 5. Limpieza con pandas

In [5]:
df_silver = df[["opportunity_id", "contact_id", "role"]].copy()
df_silver["role"] = df_silver["role"].str.strip().str.lower()
df_silver.head()

,opportunity_id,contact_id,role
0,OPP-0001114,CON-0013934,decision_maker
1,OPP-0000426,CON-0013348,decision_maker
2,OPP-0001010,CON-0014170,influencer
3,OPP-0000393,CON-0008739,end_user
4,OPP-0002735,CON-0002228,technical


## 6. Validar antes de escribir

In [6]:
assert len(df_silver) == len(df)
assert not df_silver.duplicated(subset=["opportunity_id", "contact_id"]).any()
assert df_silver["opportunity_id"].isin(opportunities["opportunity_id"]).all()
assert df_silver["contact_id"].isin(contacts["contact_id"]).all()
print("OK:", len(df_silver), "filas listas para silver")

OK: 6000 filas listas para silver


## 7. Escribir en `silver.crm__opportunity_contacts`

In [7]:
df_silver["_silver_loaded_at"] = pd.Timestamp.utcnow()

df_silver.to_sql(
    "crm__opportunity_contacts",
    engine,
    schema="silver",
    if_exists="replace",
    index=False,
    method="multi",
    chunksize=2000,
)
print("Escrito en silver.crm__opportunity_contacts")

Escrito en silver.crm__opportunity_contacts


## 8. Verificar

In [8]:
check = pd.read_sql("SELECT * FROM silver.crm__opportunity_contacts LIMIT 5", engine)
print(pd.read_sql("SELECT count(*) AS filas FROM silver.crm__opportunity_contacts", engine))
check

   filas
0   6000


,opportunity_id,contact_id,role,_silver_loaded_at
0,OPP-0001114,CON-0013934,decision_maker,2026-07-17 15:17:36.438320+00:00
1,OPP-0000426,CON-0013348,decision_maker,2026-07-17 15:17:36.438320+00:00
2,OPP-0001010,CON-0014170,influencer,2026-07-17 15:17:36.438320+00:00
3,OPP-0000393,CON-0008739,end_user,2026-07-17 15:17:36.438320+00:00
4,OPP-0002735,CON-0002228,technical,2026-07-17 15:17:36.438320+00:00
